# 🏆 مشروع توقع كأس العالم 2026
## المرحلة الرابعة: التحديث التلقائي Live (Live Update Engine)

### ترتيب تشغيل الـ Notebooks:
```
01_data_collection      → جمع البيانات
02_feature_engineering  → بناء الـ Features
04_live_update_engine   → هنا ⬅️ (قبل التدريب!)
03_model_training       → تدريب النموذج على أحدث بيانات
```

### ليه الـ 04 بييجي قبل الـ 03؟
> عشان النموذج يتدرب على **أحدث ELO** و**يعرف مين خرج فعلاً** من البطولة — مش على بيانات قديمة

### ما بيعمله:
> 1. يجيب آخر `results.csv` من GitHub (بيتحدث يومياً)
> 2. يحسب مين خرج من دور المجموعات + أدوار الإقصاء
> 3. يحدّث الـ ELO بكل نتائج WC 2026 الفعلية
> 4. يحفظ الملفات عشان الـ `03` ياخد منها

> ⚠️ النتائج بتاخد من ساعات لحد يوم تظهر على GitHub بعد المباراة — لكن الكود بيحدّث نفسه تلقائياً كل ما تشغّله


## 📦 1. استيراد المكتبات وتحميل البيانات

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

teams_df = pd.read_csv("data/raw/teams.csv")

TEAM_NAME_MAP = {
    "Holland":"Netherlands","Korea Republic":"South Korea",
    "IR Iran":"Iran","USA":"United States",
    "Côte d'Ivoire":"Ivory Coast","Czechia":"Czech Republic",
}
def normalize_team(name):
    if pd.isna(name): return name
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

teams_df["team_name"] = teams_df["team_name"].apply(normalize_team)

print(f"✅ جاهزين — تاريخ التشغيل: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"   عدد فرق WC 2026: {len(teams_df)}")


✅ جاهزين — تاريخ التشغيل: 2026-07-01 02:58
   عدد فرق WC 2026: 48


## 🔄 2. جلب آخر نتائج WC 2026 (مباشرة من GitHub)

> هنا السحر — كل ما تشغّل الـ cell ده، بيجيب أحدث نسخة من الداتا أوتوماتيك


In [2]:
BASE_MJ = "https://raw.githubusercontent.com/martj42/international_results/master/"

print("⏳ جاري جلب أحدث نتائج WC 2026...")
results_live = pd.read_csv(BASE_MJ + "results.csv", parse_dates=["date"])

for col in ["home_team","away_team"]:
    results_live[col] = results_live[col].apply(normalize_team)

# مباريات كأس العالم 2026 فقط
wc26 = results_live[
    (results_live["tournament"] == "FIFA World Cup") &
    (results_live["date"] >= "2026-01-01")
].copy().sort_values("date").reset_index(drop=True)

wc26_completed = wc26.dropna(subset=["home_score","away_score"])
wc26_pending   = wc26[wc26["home_score"].isna()]

print(f"\n✅ تم التحديث: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"   مباريات WC2026 الكلية   : {len(wc26)}")
print(f"   مباريات منتهية         : {len(wc26_completed)}")
print(f"   مباريات لسه ماجتش       : {len(wc26_pending)}")
print(f"\n   آخر مباراة مسجّلة بنتيجة: {wc26_completed['date'].max().date()}")


⏳ جاري جلب أحدث نتائج WC 2026...

✅ تم التحديث: 2026-07-01 02:58
   مباريات WC2026 الكلية   : 89
   مباريات منتهية         : 76
   مباريات لسه ماجتش       : 13

   آخر مباراة مسجّلة بنتيجة: 2026-06-29


## 🚪 3. تحديد الفرق التي خرجت من البطولة فعلياً

> منطق التحديد:
> - **دور المجموعات:** أقل فريقين نقاط في كل مجموعة (لو المجموعة خلصت بالكامل)
> - **أدوار الإقصاء:** أي فريق خسر مباراة KO = خرج فوراً (لا يوجد تعادل ممكن)


In [3]:
# ── الفرق الـ 48 المشاركة في WC 2026 ─────────────────────
# (بنستخدمها عشان نعرف مين مش ظهر في دور الـ32)
WC2026_ALL_TEAMS = {
    "Mexico","South Africa","South Korea","Canada","Bosnia and Herzegovina",
    "United States","Paraguay","Qatar","Switzerland","New Zealand","Ecuador",
    "Iran","Senegal","Netherlands","England","Scotland","France","Australia",
    "Germany","Japan","Morocco","Argentina","Belgium","Brazil","Spain",
    "Portugal","Croatia","Uruguay","Colombia","Chile","Ivory Coast","Norway",
    "Egypt","Ghana","Algeria","Tunisia","DR Congo","Saudi Arabia","Turkey",
    "Serbia","Poland","Ukraine","Costa Rica","Honduras","Jamaica","Haiti",
    "Panama","Peru","Uzbekistan","Jordan","Cape Verde","Austria","Sweden",
    "Czechia","Cuba","Curaçao","DR Congo","Iraq","Indonesia","Venezuela"
}
WC2026_ALL_TEAMS = {normalize_team(t) for t in WC2026_ALL_TEAMS}

# تاريخ بداية دور الـ32
KO_START_DATE = pd.Timestamp("2026-06-28")

def detect_eliminated_teams(wc26_results):
    """
    المنطق الصح لنظام WC 2026 (48 فريق):
    
    - دور المجموعات: أول 2 من كل مجموعة (24) + أحسن 8 توالت (8) = 32 متأهل
    - بدل ما نحسب النقاط ونفهم نظام التوالت المعقد:
      نقول ببساطة: "لو الفريق ظهر في جدول دور الـ32 = متأهل"
    
    - دور الإقصاء: أي خسارة = خروج فوري (مفيش تعادل في الـKO)
    - التعادل في الوقت الأصلي → ركلات ترجيح (مش مسجّلة في GitHub فوراً)
    """
    # ── مباريات دور الـ32 فصاهم ────────────────────────
    ko_all       = wc26_results[wc26_results["date"] >= KO_START_DATE]
    ko_completed = ko_all.dropna(subset=["home_score","away_score"])

    # الفرق اللي ظهرت في دور الـ32 = متأهلة من المجموعات
    qualified_r32 = (
        set(ko_all["home_team"].dropna()) |
        set(ko_all["away_team"].dropna())
    )

    # الفرق اللي خرجت من المجموعات = مش ظهرت في دور الـ32
    eliminated_groups = WC2026_ALL_TEAMS - qualified_r32

    # الفرق اللي خسرت في الـKO (نتيجة حاسمة مش تعادل)
    eliminated_ko = set()
    for _, m in ko_completed.iterrows():
        hs, as_ = m["home_score"], m["away_score"]
        if hs != as_:  # فيه فائز واضح — مش تعادل
            loser = m["away_team"] if hs > as_ else m["home_team"]
            eliminated_ko.add(loser)

    # ⚠️ التعادل في الـKO = ركلات ترجيح
    # GitHub بياخد وقت يسجّل الفائز — النظام بيعامله كـ "لسه في المنافسة"
    # بمجرد ما الداتا تتحدث، الفائز هيظهر في المباراة الجاية

    all_eliminated = eliminated_groups | eliminated_ko
    still_alive    = qualified_r32 - eliminated_ko

    return qualified_r32, all_eliminated, eliminated_groups, eliminated_ko, still_alive

# ── تشغيل ────────────────────────────────────────────────
qualified_r32, all_eliminated, elim_groups, elim_ko, still_alive = \
    detect_eliminated_teams(wc26)

print(f"✅ تحليل البطولة ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print(f"\n{'='*45}")
print(f"  فرق WC 2026 الكلية         : 48")
print(f"  تأهّلوا لدور الـ32          : {len(qualified_r32)}")
print(f"  خرجوا من المجموعات         : {len(elim_groups)}")
print(f"  خرجوا من دور الـ32          : {len(elim_ko)}")
print(f"  لسه في المنافسة            : {len(still_alive)}")
print(f"{'='*45}")

print(f"\n❌ خرجوا من المجموعات ({len(elim_groups)}):")
for t in sorted(elim_groups): print(f"   {t}")

if elim_ko:
    print(f"\n❌ خرجوا من دور الـ32 ({len(elim_ko)}):")
    for t in sorted(elim_ko): print(f"   {t}")

print(f"\n✅ لسه في المنافسة ({len(still_alive)}):")
for t in sorted(still_alive): print(f"   {t}")


✅ تحليل البطولة (2026-07-01 02:59)

  فرق WC 2026 الكلية         : 48
  تأهّلوا لدور الـ32          : 32
  خرجوا من المجموعات         : 27
  خرجوا من دور الـ32          : 2
  لسه في المنافسة            : 30

❌ خرجوا من المجموعات (27):
   Chile
   Costa Rica
   Cuba
   Curaçao
   Czech Republic
   Haiti
   Honduras
   Indonesia
   Iran
   Iraq
   Jamaica
   Jordan
   New Zealand
   Panama
   Peru
   Poland
   Qatar
   Saudi Arabia
   Scotland
   Serbia
   South Korea
   Tunisia
   Turkey
   Ukraine
   Uruguay
   Uzbekistan
   Venezuela

❌ خرجوا من دور الـ32 (2):
   Japan
   South Africa

✅ لسه في المنافسة (30):
   Algeria
   Argentina
   Australia
   Austria
   Belgium
   Bosnia and Herzegovina
   Brazil
   Canada
   Cape Verde
   Colombia
   Croatia
   DR Congo
   Ecuador
   Egypt
   England
   France
   Germany
   Ghana
   Ivory Coast
   Mexico
   Morocco
   Netherlands
   Norway
   Paraguay
   Portugal
   Senegal
   Spain
   Sweden
   Switzerland
   United States


## ✍️ 4. Manual Overrides — ركلات الترجيح اللي لسه مش في GitHub

> GitHub بياخد ساعات لحد يوم يسجّل نتيجة ركلات الترجيح  
> الـ cell ده بيخليك تضيف الفرق يدوياً مؤقتاً  
> **بمجرد ما GitHub يتحدث — الكود التاني هيشتغل صح أوتوماتيك ومش محتاج تعديل**


In [4]:
# ── Manual Overrides — للفرق اللي خرجت بركلات الترجيح ──
# بتحدّثه يدوياً لحد ما GitHub يسجّل النتيجة أوتوماتيك
# بمجرد ما GitHub يتحدث — الكود بيشتغل صح من غير ما تعدّل هنا

import os, csv

MANUAL_OVERRIDES_PATH = "data/processed/manual_overrides.csv"

# ── أضف هنا الفرق اللي خرجت بركلات الترجيح ─────────────
MANUAL_ELIMINATED = {
    "Germany"    : "penalties_loss_vs_Paraguay",
    "Netherlands": "penalties_loss_vs_Morocco",
    # أضف هنا أي فريق خرج بركلات ترجيح ولسه مش في GitHub
    # "France"   : "penalties_loss_vs_XXX",
}

# حفظ الـ overrides في ملف CSV
os.makedirs("data/processed", exist_ok=True)
with open(MANUAL_OVERRIDES_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["team","reason","added_date"])
    writer.writeheader()
    for team, reason in MANUAL_ELIMINATED.items():
        writer.writerow({
            "team"      : team,
            "reason"    : reason,
            "added_date": datetime.now().strftime("%Y-%m-%d"),
        })

print(f"✅ Manual Overrides محفوظة: {len(MANUAL_ELIMINATED)} فريق")
for team, reason in MANUAL_ELIMINATED.items():
    print(f"   ❌ {team} — {reason}")
print(f"\n⚠️  بمجرد ما GitHub يتحدث — الكود هيشتغل أوتوماتيك ومش محتاج تعدّل هنا")


✅ Manual Overrides محفوظة: 2 فريق
   ❌ Germany — penalties_loss_vs_Paraguay
   ❌ Netherlands — penalties_loss_vs_Morocco

⚠️  بمجرد ما GitHub يتحدث — الكود هيشتغل أوتوماتيك ومش محتاج تعدّل هنا


## 🔍 4. تحقق يدوي — هل ألمانيا ظاهرة كمُقصاة؟

> طالما الداتا اتحدثت على GitHub، النتيجة هتظهر هنا أوتوماتيك من غير أي تعديل في الكود


In [5]:
# ── دمج الـ Manual Overrides مع الـ Auto Detection ────────
manual_elim = set(MANUAL_ELIMINATED.keys())

# الفرق الخارجة النهائية = Auto + Manual
all_eliminated_final = all_eliminated | manual_elim
still_alive_final    = still_alive - manual_elim

print(f"✅ بعد دمج Manual + Auto:")
print(f"   خرجوا أوتوماتيك  : {len(all_eliminated)}")
print(f"   خرجوا يدوياً     : {len(manual_elim)}")
print(f"   إجمالي الخارجين  : {len(all_eliminated_final)}")
print(f"   لسه في المنافسة  : {len(still_alive_final)}")

print(f"\n❌ كل الفرق الخارجة:")
for t in sorted(all_eliminated_final):
    source = "يدوي ⚠️" if t in manual_elim else "أوتوماتيك ✅"
    print(f"   {t} ({source})")

print(f"\n✅ لسه في المنافسة ({len(still_alive_final)}):")
for t in sorted(still_alive_final):
    print(f"   {t}")

# ── تحقق من فريق معين ───────────────────────────────────
check_team = "Germany"
print(f"\n--- {check_team} ---")
if check_team in manual_elim:
    print(f"❌ خرجت (مسجّلة يدوياً — ركلات ترجيح)")
    print(f"   السبب: {MANUAL_ELIMINATED.get(check_team)}")
    print(f"   ⚠️  بمجرد ما GitHub يتحدث — هتُحذف من هنا أوتوماتيك")
elif check_team in elim_ko:
    print(f"❌ خرجت أوتوماتيك من دور الـKO")
elif check_team in elim_groups:
    print(f"❌ خرجت من دور المجموعات")
elif check_team in still_alive_final:
    print(f"✅ لسه في المنافسة")


✅ بعد دمج Manual + Auto:
   خرجوا أوتوماتيك  : 29
   خرجوا يدوياً     : 2
   إجمالي الخارجين  : 31
   لسه في المنافسة  : 28

❌ كل الفرق الخارجة:
   Chile (أوتوماتيك ✅)
   Costa Rica (أوتوماتيك ✅)
   Cuba (أوتوماتيك ✅)
   Curaçao (أوتوماتيك ✅)
   Czech Republic (أوتوماتيك ✅)
   Germany (يدوي ⚠️)
   Haiti (أوتوماتيك ✅)
   Honduras (أوتوماتيك ✅)
   Indonesia (أوتوماتيك ✅)
   Iran (أوتوماتيك ✅)
   Iraq (أوتوماتيك ✅)
   Jamaica (أوتوماتيك ✅)
   Japan (أوتوماتيك ✅)
   Jordan (أوتوماتيك ✅)
   Netherlands (يدوي ⚠️)
   New Zealand (أوتوماتيك ✅)
   Panama (أوتوماتيك ✅)
   Peru (أوتوماتيك ✅)
   Poland (أوتوماتيك ✅)
   Qatar (أوتوماتيك ✅)
   Saudi Arabia (أوتوماتيك ✅)
   Scotland (أوتوماتيك ✅)
   Serbia (أوتوماتيك ✅)
   South Africa (أوتوماتيك ✅)
   South Korea (أوتوماتيك ✅)
   Tunisia (أوتوماتيك ✅)
   Turkey (أوتوماتيك ✅)
   Ukraine (أوتوماتيك ✅)
   Uruguay (أوتوماتيك ✅)
   Uzbekistan (أوتوماتيك ✅)
   Venezuela (أوتوماتيك ✅)

✅ لسه في المنافسة (28):
   Algeria
   Argentina
   Australia
   Austria

## ⚡ 5. تحديث ELO بآخر نتائج البطولة الفعلية

> بدل ما نعتمد بس على ELO التاريخي — بنحدّثه بكل مباراة لعبت في WC 2026 لحد دلوقتي


In [6]:
elo_current = pd.read_csv("data/processed/elo_current.csv")
elo_current["team"] = elo_current["team"].apply(normalize_team)
elo_dict = dict(zip(elo_current["team"], elo_current["elo"]))

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def update_elo_live(elo_dict, wc26_completed_matches, k=40):
    """
    تحديث الـ ELO بكل مباراة فعلية اتلعبت في WC 2026.
    k=40 (أعلى من المعتاد) — لأن كأس العالم أهم بطولة، التأثير أكبر
    """
    updated = elo_dict.copy()
    for _, m in wc26_completed_matches.sort_values("date").iterrows():
        h, a = m["home_team"], m["away_team"]
        hs, as_ = m["home_score"], m["away_score"]

        ra = updated.get(h, 1500)
        rb = updated.get(a, 1500)

        w_h = 1.0 if hs>as_ else (0.0 if hs<as_ else 0.5)
        w_a = 1.0 - w_h

        e_h = expected_score(ra, rb)  # ملعب محايد غالباً في WC
        e_a = 1 - e_h

        updated[h] = ra + k * (w_h - e_h)
        updated[a] = rb + k * (w_a - e_a)
    return updated

elo_updated = update_elo_live(elo_dict, wc26_completed)

elo_updated_df = pd.DataFrame(
    list(elo_updated.items()), columns=["team","elo_live"]
).sort_values("elo_live", ascending=False).reset_index(drop=True)
elo_updated_df["rank"] = elo_updated_df.index + 1

elo_updated_df.to_csv("data/processed/elo_live_updated.csv", index=False)

print("✅ ELO اتحدّث بكل نتائج WC 2026 الفعلية")
print(f"\n🏆 أعلى 15 فريق بعد التحديث الـ Live:")
display(elo_updated_df.head(15))


✅ ELO اتحدّث بكل نتائج WC 2026 الفعلية

🏆 أعلى 15 فريق بعد التحديث الـ Live:


,team,elo_live,rank
0,Argentina,2216.332383,1
1,Spain,2211.312543,2
2,France,2195.789215,3
3,England,2128.652892,4
4,Colombia,2105.473600,5
5,Brazil,2099.570335,6
6,Norway,2077.971598,7
7,Morocco,2051.350236,8
8,Netherlands,2042.878080,9
9,Portugal,2019.215879,10


## 🎲 6. Monte Carlo محدّث — بس على الفرق المتأهلة فعلياً

> دلوقتي الفرق المُقصاة مش هتدخل المحاكاة خالص — حتى لو ELO بتاعها عالي تاريخياً


In [12]:
import joblib
from scipy.stats import poisson as sp_poisson

# تحميل النماذج المتوازنة من المرحلة الثالثة
xgb_model    = joblib.load("models/xgb_model_balanced.pkl")
poisson_home = joblib.load("models/poisson_home.pkl")
poisson_away = joblib.load("models/poisson_away.pkl")
scaler       = joblib.load("models/scaler.pkl")
FEATURE_COLS = joblib.load("models/feature_cols.pkl")

shootout_stats = pd.read_csv("data/processed/shootout_stats.csv")
shootout_stats["team"] = shootout_stats["team"].apply(normalize_team)

def get_shootout_stats(team):
    row = shootout_stats[shootout_stats["team"]==team]
    if len(row)==0: return {"shootout_win_rate":0.5}
    return {"shootout_win_rate": float(row.iloc[0]["shootout_win_rate"])}

def elo_to_win_prob(elo_diff, home_advantage=0):
    return 1 / (1 + 10 ** (-(elo_diff + home_advantage) / 400))

def simulate_with_live_elo(team_a, team_b):
    """محاكاة مباراة باستخدام الـ ELO المحدّث Live"""
    elo_a = elo_updated.get(team_a, 1500)
    elo_b = elo_updated.get(team_b, 1500)
    elo_diff = elo_a - elo_b
    p_a = elo_to_win_prob(elo_diff)

    # محاكاة بسيطة معتمدة على Poisson مع لامبدا متناسبة مع قوة الفريق
    lam_a = 1.0 + (elo_diff / 400)
    lam_b = 1.0 - (elo_diff / 400)
    lam_a = max(0.3, lam_a)
    lam_b = max(0.3, lam_b)

    gh = np.random.poisson(lam_a)
    ga = np.random.poisson(lam_b)

    if gh > ga: return team_a
    elif ga > gh: return team_b
    else:
        so_a = get_shootout_stats(team_a)["shootout_win_rate"]
        so_b = get_shootout_stats(team_b)["shootout_win_rate"]
        tot = so_a + so_b or 1
        return team_a if np.random.random() < so_a/tot else team_b

def run_monte_carlo_live(qualified_teams, n=10000):
    """محاكاة البطولة بس على الفرق المتأهلة فعلياً"""
    import random
    teams_list = sorted(qualified_teams)
    wins = {t: 0 for t in teams_list}

    for _ in range(n):
        remaining = teams_list.copy()
        random.shuffle(remaining)
        while len(remaining) > 1:
            nxt = []
            for i in range(0, len(remaining)-1, 2):
                nxt.append(simulate_with_live_elo(remaining[i], remaining[i+1]))
            if len(remaining) % 2 == 1:
                nxt.append(remaining[-1])
            remaining = nxt
        wins[remaining[0]] += 1

    return pd.DataFrame([
        {"#":i+1, "الفريق":t, "% الفوز بالكأس": round(w/n*100,2)}
        for i,(t,w) in enumerate(sorted(wins.items(), key=lambda x:-x[1]))
    ])

print(f"⏳ جاري تشغيل Monte Carlo على {len(still_alive_final)} فريق متأهل فعلياً...")
print(f"   (الفرق المُقصاة مستبعدة تلقائياً: {len(all_eliminated_final)} فريق)")

mc_live = run_monte_carlo_live(still_alive_final, n=5000)
mc_live.to_csv("data/processed/monte_carlo_live.csv", index=False)

print("\n🏆 نسب الفوز بالكأس (محدّثة بآخر نتائج WC 2026):\n")
display(mc_live.head(15))


⏳ جاري تشغيل Monte Carlo على 28 فريق متأهل فعلياً...
   (الفرق المُقصاة مستبعدة تلقائياً: 31 فريق)

🏆 نسب الفوز بالكأس (محدّثة بآخر نتائج WC 2026):



,#,الفريق,% الفوز بالكأس
0,1,Argentina,22.36
1,2,Spain,18.04
2,3,France,14.62
3,4,Colombia,8.30
4,5,England,7.14
5,6,Brazil,7.06
6,7,Norway,5.72
7,8,Morocco,3.68
8,9,Portugal,2.82
9,10,United States,2.42


## 🔁 7. سكريبت Auto-Refresh — شغّله يومياً

> الكود ده بيلخّص كل الخطوات في دالة واحدة — شغّلها كل يوم وهتاخد أحدث حالة للبطولة


In [13]:
def daily_update():
    """
    شغّلها كل يوم — بتحدّث كل حاجة أوتوماتيك:
    1. آخر نتائج WC 2026 من GitHub
    2. الفرق الخارجة (Auto + Manual Overrides)
    3. ELO محدّث
    4. حفظ الملفات للـ notebook 03
    """
    print(f"🔄 تحديث يومي — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 50)

    # 1. جلب آخر نتائج
    results_new = pd.read_csv(BASE_MJ + "results.csv", parse_dates=["date"])
    for col in ["home_team","away_team"]:
        results_new[col] = results_new[col].apply(normalize_team)
    wc26_new = results_new[
        (results_new["tournament"]=="FIFA World Cup") &
        (results_new["date"]>="2026-01-01")
    ].copy()
    wc26_completed_new = wc26_new.dropna(subset=["home_score","away_score"])
    print(f"✅ مباريات منتهية: {len(wc26_completed_new)}")

    # 2. تحديد المتأهلين (Auto)
    q_r32, all_elim, elim_g, elim_k, alive = detect_eliminated_teams(wc26_new)

    # 3. دمج Manual Overrides
    # لو الفريق ظهر في مباراة جاية في GitHub = GitHub اتحدث = نشيله من Manual
    upcoming_teams = (
        set(wc26_new[wc26_new["home_score"].isna()]["home_team"].dropna()) |
        set(wc26_new[wc26_new["home_score"].isna()]["away_team"].dropna())
    )
    active_manual = {
        t: r for t, r in MANUAL_ELIMINATED.items()
        if t not in elim_k and t not in upcoming_teams
    }
    if len(active_manual) < len(MANUAL_ELIMINATED):
        auto_updated = set(MANUAL_ELIMINATED) - set(active_manual)
        print(f"🎉 GitHub اتحدث! هذه الفرق بقت أوتوماتيك: {auto_updated}")

    all_eliminated_final = all_elim | set(active_manual.keys())
    still_alive_final    = q_r32 - all_eliminated_final

    print(f"✅ متأهلين: {len(still_alive_final)} | مُقصاة: {len(all_eliminated_final)}")
    if active_manual:
        print(f"   منهم يدوياً: {set(active_manual.keys())}")

    # 4. تحديث ELO
    elo_updated_new = update_elo_live(elo_dict, wc26_completed_new)
    print(f"✅ ELO محدّث لـ {len(elo_updated_new)} فريق")

    # 5. حفظ الملفات
    pd.DataFrame(list(elo_updated_new.items()), columns=["team","elo_live"])\
      .sort_values("elo_live", ascending=False)\
      .to_csv("data/processed/elo_live_updated.csv", index=False)

    with open("data/processed/eliminated_teams.txt","w") as f:
        f.write("\n".join(sorted(all_eliminated_final)))

    print("\n💾 الملفات اتحدّثت في data/processed/")
    print("   elo_live_updated.csv")
    print("   eliminated_teams.txt")
    print("=" * 50)

    return still_alive_final, all_eliminated_final, elo_updated_new

# ── شغّل التحديث اليومي ──────────────────────────────────
still_alive_final, all_eliminated_final, elo_updated = daily_update()


🔄 تحديث يومي — 2026-07-01 03:58
✅ مباريات منتهية: 76
✅ متأهلين: 28 | مُقصاة: 31
   منهم يدوياً: {'Germany', 'Netherlands'}
✅ ELO محدّث لـ 217 فريق

💾 الملفات اتحدّثت في data/processed/
   elo_live_updated.csv
   eliminated_teams.txt


## ✅ ملخص

### الملفات الجديدة:
| الملف | المحتوى |
|-------|---------|
| `data/processed/elo_live_updated.csv` | ELO محدّث بآخر نتائج البطولة |
| `data/processed/eliminated_teams.txt` | قائمة الفرق المُقصاة |
| `data/processed/monte_carlo_live.csv` | نسب الفوز محدّثة على المتأهلين فقط |

### كيفية الاستخدام:
> شغّل `daily_update()` كل يوم — هيجيب أحدث نتائج تلقائياً من GitHub ويحدّث كل حاجة.  
> ⚠️ النتائج بتاخد من ساعات لحد يوم تظهر على GitHub بعد المباراة الفعلية.

### الخطوة الجاية: المرحلة الخامسة — Dashboard احترافي بـ Streamlit للنشر على LinkedIn/Kaggle 🚀
